In [ ]:
# -----------------------------------------------
# 🔆 光源调整模块可视化 (for LAM)
# -----------------------------------------------
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from torchvision.utils import make_grid
from torchvision.transforms.functional import to_pil_image

# ------------------------------------------------
# Step 1. 加载模型
# ------------------------------------------------
# 假设模型定义在 my_model.py 中，或者你可以直接在上方cell定义了 MyNet2_5
from model import MyNet2_5  # 若你在同文件中定义，可跳过这行
model = MyNet2_5(base_channels=16).cuda()
ckpt_path = "checkpoint.pth"  # 你的模型路径
state = torch.load(ckpt_path, map_location='cuda')
model.load_state_dict(state)
model.eval()

# ------------------------------------------------
# Step 2. 加载输入图像
# ------------------------------------------------
from PIL import Image
from torchvision import transforms

img_path = "test_image.jpg"  # 输入待测试图像
img = Image.open(img_path).convert("RGB")

transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])
x = transform(img).unsqueeze(0).cuda()  # (1,3,H,W)

# ------------------------------------------------
# Step 3. 前向推理并获取中间结果
# ------------------------------------------------
with torch.no_grad():
    out, alpha, params, light_map = model.forward_with_light(x)
    # 注意：确保你的 MyNet2_5.forward_with_light() 返回这些变量

# ------------------------------------------------
# Step 4. 可视化函数
# ------------------------------------------------
def visualize_lam_results(x, out, alpha, params, light_map, save_path=None):
    """
    x: 原图 (1,3,H,W)
    out: 输出图 (1,3,H,W)
    alpha: 光源mask (1,1,H,W)
    params: 光源参数 (1, n_lights, 11)
    light_map: 渲染光源图 (1,3,H,W)
    """
    x_np = x[0].detach().cpu()
    out_np = out[0].detach().cpu()
    light_np = light_map[0].detach().cpu()
    alpha_np = alpha[0,0].detach().cpu()

    H, W = alpha_np.shape
    n_lights = params.shape[1]

    fig, axs = plt.subplots(2, 3, figsize=(14, 8))
    axs = axs.flatten()

    # 1. 原图
    axs[0].imshow(to_pil_image(x_np))
    axs[0].set_title("Input")

    # 2. 输出图
    axs[1].imshow(to_pil_image(out_np))
    axs[1].set_title("Output (Restored)")

    # 3. 光源渲染
    axs[2].imshow(to_pil_image(light_np))
    axs[2].set_title("Rendered Light Map")

    # 4. 光源mask
    im = axs[3].imshow(alpha_np, cmap='magma')
    axs[3].set_title("Alpha (Light Mask)")
    plt.colorbar(im, ax=axs[3], fraction=0.046, pad=0.04)

    # 5. 光源参数可视化
    axs[4].imshow(to_pil_image(x_np))
    axs[4].set_title("Light Params Overlay")
    for i in range(n_lights):
        p = params[0, i].detach().cpu().numpy()
        x_pos, y_pos = p[0] * W, p[1] * H
        a, b, angle = p[2]*W, p[3]*H, p[4]
        conf = p[6]
        if conf < 0.05:  # 跳过置信度过低的光源
            continue
        color = (p[7], p[8], p[9])
        e = patches.Ellipse((x_pos, y_pos), 2*a, 2*b, 
                            angle=angle*180/3.1416, 
                            linewidth=2, edgecolor=color, facecolor='none', alpha=0.8)
        axs[4].add_patch(e)
        axs[4].scatter([x_pos], [y_pos], c=[color], s=80, marker='x')

    # 6. 融合结果 (x + alpha*light)
    merged = (1 - alpha) * x + alpha * light_map
    axs[5].imshow(to_pil_image(merged[0].detach().cpu()))
    axs[5].set_title("Alpha-guided Fusion")

    for ax in axs:
        ax.axis('off')

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()

# ------------------------------------------------
# Step 5. 调用可视化
# ------------------------------------------------
visualize_lam_results(x, out, alpha, params, light_map)
